# Chapter 3 – Quantum Probability  
*From describing probability to producing it*

This notebook builds on the circuits introduced in the previous chapter and focuses on how quantum systems work with **probability**. Rather than observing randomness alone, we begin to model uncertainty, encode it into a **quantum state**, and measure outcomes that follow from that state.

We start by comparing **classical** and **quantum randomness**, then move into probability models that evolve as a system changes. From there, we carry those probabilities into a quantum circuit, where they are represented as **amplitudes**. When measured, those amplitudes produce outcomes that reflect the distribution we encoded.

The key idea is that probability is no longer something we compute separately. It becomes part of the system itself. Once encoded, the system produces outcomes directly through **measurement**.

We then extend this by introducing **interference**. By applying additional operations, we reshape how outcomes appear without changing the underlying possibilities. This gives us a way to guide results, reinforcing some outcomes while reducing others.

Each section builds step by step, following the same pattern used throughout: `prepare`, `transform`, and `measure`. As you run the code, focus on how the system evolves and how those changes affect the results.

### Code 3-1: Classical Randomness

This example introduces classical randomness using a simple **linear congruential generator (LCG)**. The first cell defines the `lcg` function and generates a short sequence from a fixed seed. The second cell reuses that function to produce a larger dataset and visualize it in three dimensions using `plot_3d_random_values`.

**Note:** Be sure to run both cells in order. The second cell depends on the `lcg` function defined in the first.

As you run this, focus on the **structure of the points**. Although the values appear random, they form visible patterns when plotted at scale. This reflects an important idea: classical randomness is often generated by **deterministic processes**. Even when outputs look unpredictable, underlying structure can still emerge.


In [ ]:
# Linear Congruential Generator (LCG): produces a sequence from a seed
def lcg(seed, a=1664525, c=1013904223, m=2**32):
    seed = (a * seed + c) % m
    return seed

# Example: generate a short sequence from a fixed seed
seed = 42

for _ in range(5):
    seed = lcg(seed)
    print(seed)

In [ ]:
# Requires: lcg function from previous cell

import numpy as np
import matplotlib.pyplot as plt

# Plot values as (n, n+1, n+2) in 3D.
# Uses consistent styling so differences reflect the data.
def plot_3d_random_values(data, title, is_quantum=False):
    x, y, z = data[0::3], data[1::3], data[2::3]

    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection="3d")

    # 'inferno' stays away from white/yellow peaks more than 'plasma'
    # 'viridis' is classic for the quantum 'mist'
    color_map = "viridis" if is_quantum else "inferno"

    # THE "MAX NOTCH" SETTINGS:
    # s=15.0: Significant ink presence
    # alpha=1.0: Zero transparency to ensure maximum saturation
    # edgecolors='black', linewidth=0.2: Adds a 'border' so dots don't vanish
    ax.scatter(x, y, z,
               s=15.0,
               c=z,
               cmap=color_map,
               alpha=1.0,
               edgecolors='black',
               linewidth=0.2)

    ax.set_title(title, fontsize=20, pad=30)

    # Clean transparent panes
    ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))

    # Standard labels (No Bold)
    ax.set_xlabel("Value n", fontsize=14, labelpad=12)
    ax.set_ylabel("Value n + 1", fontsize=14, labelpad=12)
    ax.set_zlabel("Value n + 2", fontsize=14, labelpad=12)

    ax.view_init(elev=15, azim=65)

    plt.tight_layout()
    plt.show()

# Generate values using the LCG with RANDU parameters
def generate_sequence(seed, count):
    values = []

    for _ in range(count):
        # RANDU parameters highlight structure when visualized
        seed = lcg(seed, a=65539, c=0, m=2**31)
        values.append(seed / (2**31))

    return np.array(values)

# Generate data and visualize
n_points = 5000
data = generate_sequence(seed=1, count=n_points * 3)

plot_3d_random_values(
    data,
    title="Classical Randomness: RANDU Spectral Test"
)

### Code 3-2: Quantum Randomness

This example explores quantum randomness using a simple circuit built with `QuantumCircuit`. We place multiple qubits into superposition using the `h` gate and then measure them repeatedly to generate random bitstrings. These results are converted into numeric values and visualized using `plot_3d_random_values`.

**Note:** Run the `pip install` cell first to ensure `qiskit` and related packages are available. This cell also reuses the plotting function from the previous example.

As you run this, focus on the **distribution of the points**. Unlike the classical case, the values form a more uniform cloud without visible structure. This reflects a key difference. Quantum randomness emerges from **measurement of a superposed state**, not from a deterministic sequence.


In [ ]:
!pip -q install qiskit qiskit-aer pylatexenc

In [ ]:
# Requires: plot_3d_random_values function from previous cell

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer

# Generate random values from repeated quantum measurements
def get_quantum_random_numbers(n_points):
    n_qubits = 10  # Controls resolution (2**10 possible values)

    # Build a circuit that places all qubits into superposition
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))
    qc.measure_all()

    # Run the circuit on a simulator
    simulator = Aer.get_backend("qasm_simulator")
    compiled = transpile(qc, simulator)

    # Collect individual measurement results (bitstrings)
    job = simulator.run(compiled, shots=n_points * 3, memory=True)
    bitstrings = job.result().get_memory()

    # Convert bitstrings to normalized values between 0 and 1
    values = [
        int(bits, 2) / (2**n_qubits - 1)
        for bits in bitstrings
    ]

    return np.array(values)

# Generate data and visualize
n_points = 5000
q_data = get_quantum_random_numbers(n_points)

plot_3d_random_values(
    q_data,
    title="Quantum Randomness: Structureless Cloud"
)

### Code 3-3: Classical Die Roll

This example builds a simple **probability model** for a six-sided die and compares it to a simulation. We start by defining a uniform distribution, then generate outcomes using `random.randint` and summarize them with `Counter` to approximate the expected probabilities.

Next, we update the model based on new information. By restricting outcomes to even numbers, we recompute the distribution to reflect the updated state.

As you run this, focus on how the **model changes with new information**. The probabilities are not fixed. They evolve based on what we know about the system. This sets up the idea that probability models can adapt, which we will later carry into a quantum state.

In [ ]:
# Code 3-3: Classical die probability model and simulation

import random
from collections import Counter

# Step 1: Define the probability model (uniform die)
die = {i: 1/6 for i in range(1, 7)}
print("Initial model:", die)

# Step 2: Simulate repeated rolls
rolls = [random.randint(1, 6) for _ in range(1000)]
counts = Counter(rolls)
simulation = {k: counts[k] / 1000 for k in sorted(counts)}
print("Simulation:", simulation)

# Step 3: Update the model (given the result is even)
updated = {k: v for k, v in die.items() if k % 2 == 0}
total = sum(updated.values())
updated = {k: v / total for k, v in updated.items()}
print("Updated model (even):", updated)

### Code 3-4: Quantum Rolling a Die

This example builds a simple **quantum die** using `QuantumCircuit`. We place three qubits into superposition with the `h` gate, then measure them repeatedly using `AerSimulator`. Each measurement produces a three-bit string, which we map to die faces.

**Note:** Be sure to run the earlier `pip install` cell for `qiskit` and related packages before executing this code.

As you run this, focus on how **measurement produces outcomes from a quantum state**. The system generates multiple possibilities at once, and measurement selects one result each time. By mapping valid bitstrings to die faces, we recover a distribution that approximates a fair die. This connects the idea of superposition to generating outcomes from a quantum system.

In [ ]:
# Code 3-4: Quantum die using 3 qubits

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from collections import Counter

# Step 1: Create a quantum circuit with 3 qubits
qc = QuantumCircuit(3, 3)
qc.h([0, 1, 2])          # Put all qubits into superposition
qc.measure([0, 1, 2], [0, 1, 2])

# Step 2: Run the circuit multiple times
sim = AerSimulator()
result = sim.run(qc, shots=2000).result()
counts = result.get_counts()

print("Raw bitstrings:", counts)

# Step 3: Map bitstrings to die faces (6 out of 8 outcomes)
mapping = {
    "000": 1,
    "001": 2,
    "010": 3,
    "011": 4,
    "100": 5,
    "101": 6
}

die_counts = Counter()

for bits, count in counts.items():
    if bits in mapping:
        die_counts[mapping[bits]] += count

print("Quantum die:", dict(sorted(die_counts.items())))
print("Unused outcomes:", counts.get("110", 0) + counts.get("111", 0))

### Code 3-5: Quantum High–Low

This example brings together the ideas from this chapter in a single workflow. We start with a simple card game and build a **probability model** based on the remaining deck. That model is then carried into a quantum system, where it is encoded as amplitudes and expressed through measurement.

**Note:** Be sure to run the `pip install` cell before this section. This example depends on `qiskit` and related packages.

The steps follow a clear progression. We define the system, compute classical probabilities, encode those probabilities into a quantum state, and then measure the results. Finally, we take a single sample to produce one outcome.

As you run this, focus on how the system evolves. The probabilities are not just calculated. They are embedded in the quantum state and used to generate outcomes. This is the shift from describing probability to producing it.

In [ ]:
!pip -q install qiskit qiskit-aer pylatexenc

#### Code 3-5-1: Build the card model and draw a card

This step defines the system using a simple **deck of cards**. We create the deck, shuffle it with `random.shuffle`, and draw a single card using `pop` to represent the current state.

As you run this, focus on what changes. The drawn card is fixed, and the remaining deck defines what is still possible. This becomes the basis for computing probabilities in the next step.

In [ ]:
from collections import Counter
import random

# Define ranks and suits
RANKS = ["2", "3", "4", "5", "6", "7", "8", "9", "10",
         "J", "Q", "K", "A"]
SUITS = ["Hearts", "Diamonds", "Clubs", "Spades"]

# Map ranks to numeric values for comparison
RANK_VALUES = {rank: i + 2 for i, rank in enumerate(RANKS)}

# Create and shuffle the deck
deck = [(rank, suit) for suit in SUITS for rank in RANKS]
random.shuffle(deck)

# Draw the current card and remove it from the deck
current = deck.pop(0)

print("Current card:", current)
print("Cards remaining:", len(deck))

#### Code 3-5-2: Compute the classical probabilities

This step computes a **probability model** from the remaining deck. For each card, we compare its value to the current card and count outcomes using `counts`.

The results are normalized to produce probabilities for higher, lower, and tie.

As you run this, focus on how the **distribution reflects the current state**. The probabilities are not fixed. They are determined by what remains in the deck and will change as the system evolves.

In [ ]:
# Count how many remaining cards are higher, lower, or equal
counts = {"higher": 0, "lower": 0, "tie": 0}

current_value = RANK_VALUES[current[0]]

for rank, suit in deck:
    value = RANK_VALUES[rank]

    if value > current_value:
        counts["higher"] += 1
    elif value < current_value:
        counts["lower"] += 1
    else:
        counts["tie"] += 1

# Convert counts into probabilities
total = len(deck)

probs = {
    "higher": counts["higher"] / total,
    "lower": counts["lower"] / total,
    "tie": counts["tie"] / total,
}

print("Classical probabilities:", probs)

#### Code 3-5-2: Compute the classical probabilities

This step computes a **probability model** from the remaining deck. For each card, we compare its value to the current card and count outcomes using `counts`.

The results are normalized to produce probabilities for higher, lower, and tie.

As you run this, focus on how the **distribution reflects the current state**. The probabilities are not fixed. They are determined by what remains in the deck and will change as the system evolves.

In [ ]:
import math
from qiskit import QuantumCircuit

# Convert probabilities into amplitudes (square roots)
amplitudes = [
    math.sqrt(probs["higher"]),  # |00> → higher
    math.sqrt(probs["lower"]),   # |01> → lower
    math.sqrt(probs["tie"]),     # |10> → tie
    0.0                          # |11> → unused
]

# Create a 2-qubit circuit for state preparation
state_qc = QuantumCircuit(2)

# Initialize the qubits to match the desired distribution
# This prepares the system so measurement follows our probabilities
state_qc.initialize(amplitudes, [0, 1])

# Visualize the prepared state
state_qc.draw('mpl')

#### Code 3-5-4: Measure the quantum state

This step measures the prepared quantum state many times using `AerSimulator`. Each run produces a bitstring, and those bitstrings are mapped back to higher, lower, and tie.

As you run this, compare the **quantum probabilities** with the classical model from the previous step. They should be close, with small differences caused by finite sampling. The state was prepared using the classical probabilities, and measurement reveals that distribution.

In [ ]:
# Measure the quantum circuit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

shots = 2000

# Create a new circuit with classical bits for measurement
qc = QuantumCircuit(2, 2)

# Bring in the prepared quantum state
qc.compose(state_qc, inplace=True)

# Add measurement (this is now required)
qc.measure([0, 1], [0, 1])

sim = AerSimulator()
result = sim.run(qc, shots=shots).result()
counts = result.get_counts()

# Convert measured bitstrings into probabilities
quantum_probs = {
    "higher": counts.get("00", 0) / shots,
    "lower": counts.get("01", 0) / shots,
    "tie": counts.get("10", 0) / shots,
}

# Compare classical and quantum results
comparison = {
    outcome: {
        "classical": probs[outcome],
        "quantum": quantum_probs[outcome]
    }
    for outcome in ["higher", "lower", "tie"]
}

print(comparison)

#### Code 3-5-5: Make a quantum prediction

This step runs the measured circuit once using `shots=1`. The result is a single bitstring, which we map to higher, lower, or tie.

As you run this, focus on the difference between a **distribution** and a **single outcome**. The distribution tells us what should happen over many runs. A single measurement gives one result from the prepared state.

In [ ]:
# Run the circuit once (single measurement)
sample_result = sim.run(qc, shots=1).result()
sample_counts = sample_result.get_counts()

bitstring = list(sample_counts.keys())[0]

mapping = {
    "00": "higher",
    "01": "lower",
    "10": "tie",
    "11": "unused"
}

prediction = mapping[bitstring]

print("Quantum prediction:", prediction)

### Code 3-6: Quantum Interference

This example extends the quantum high–low model by introducing **interference**. We start from the same prepared quantum state and apply an additional operation before measurement. This does not change the possible outcomes, but it changes how the amplitudes combine.

**Note:** Re-run the cells from Code 3-5, including the `pip install` step, before executing this section. This example depends on the prepared state and earlier variables.

The process follows the same pattern. We reuse the state, apply a transformation, and measure the result. The difference is what the transformation does. Instead of representing probability, the system now reshapes it. As you run this, focus on how the distribution changes, not just the individual values.

#### Code 3-6-1: Apply interference to the prepared state

This step reuses the prepared quantum state and applies a `h` gate to one qubit before measurement. This changes how the **amplitudes combine** across outcomes.

As you run this, focus on the circuit. The state is the same, but the added operation changes how it will behave when measured.

In [ ]:
from qiskit import QuantumCircuit

# Start from the prepared high-low quantum state
interference_qc = QuantumCircuit(2, 2)
interference_qc.compose(state_qc, inplace=True)

# Apply an operation before measurement
# This changes how amplitudes combine
interference_qc.h(0)

# Measure both qubits
interference_qc.measure([0, 1], [0, 1])

# Visualize the new circuit
interference_qc.draw("mpl")

#### Code 3-6-2: Measure and compare results

This step measures the updated circuit and compares the results to the earlier distribution. We collect outcomes in `interference_counts` and convert them into probabilities.

As you run this, focus on the **change in distribution**. The outcomes remain the same, but their likelihood shifts. The optional `plot_comparison` function helps visualize this difference.

In [ ]:
# Helper function to visualize probability changes (not shown in text)
def plot_comparison(before, after, outcomes):
    import matplotlib.pyplot as plt

    x = range(len(outcomes))
    width = 0.35

    plt.figure(figsize=(8, 4))

    bars_before = plt.bar([i - width/2 for i in x], before, width, label="Before")
    bars_after = plt.bar([i + width/2 for i in x], after, width, label="After")

    # Add value labels
    for bar in bars_before:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            height,
            f"{height:.3f}",
            ha="center",
            va="bottom"
        )

    for bar in bars_after:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            height,
            f"{height:.3f}",
            ha="center",
            va="bottom"
        )

    plt.xticks(x, outcomes)
    plt.ylabel("Probability")
    plt.title("Probability Before and After Interference")
    plt.ylim(0, 1.05)
    plt.legend()
    plt.show()


# Measure the circuit after interference
result = sim.run(interference_qc, shots=shots).result()
interference_counts = result.get_counts()

interference_probs = {
    "higher": interference_counts.get("00", 0) / shots,
    "lower": counts.get("01", 0) / shots,
    "tie": interference_counts.get("10", 0) / shots,
}

# Compare before and after interference
outcomes = ["higher", "lower", "tie"]
before = [quantum_probs[outcome] for outcome in outcomes]
after = [interference_probs[outcome] for outcome in outcomes]

print("Before interference:", quantum_probs)
print("After interference:", interference_probs)

# Optional visualization
plot_comparison(before, after, outcomes)